# TP Ciencia de Datos 2026 — Actividad 2
## Scraping con LLM (Gemini) en `books.toscrape.com`

**Objetivo**: extraer **5 libros** con `título`, `género`, `precio` (float) y `stock` (int) usando un modelo de lenguaje grande para inferir los datos a partir del HTML.

**Modelo utilizado**: `gemini-3.1-flash-lite` (con fallback) vía API REST de Google AI Studio.

## Arquitectura del pipeline ETL

```
HTML  →  Markdown  →  JSON (LLM)  →  DataFrame  →  CSV
 (1)      (2)             (3)              (4)        (5)
```

| Paso | Qué hace | Herramienta |
|---|---|---|
| 1 | Descarga el HTML del libro | `requests` |
| 2 | Convierte HTML → Markdown limpio | `markdownify` (rol equivalente a **Crawl4AI**) |
| 3 | Infiere `{titulo, genero, precio, stock}` como JSON | API REST de **Gemini** (rol equivalente a **ScrapeGraphAI**) |
| 4 | Acumula la lista de dicts en un DataFrame **una sola vez** al final | `pandas` |
| 5 | Valida tipos / nulos y exporta | `pandas.to_csv` |

## Decisiones técnicas

| Decisión | Justificación |
|---|---|
| Sustituir Crawl4AI por `requests + markdownify` | Crawl4AI requiere Playwright + Chromium (instalación pesada en Windows local). Como el sitio es estático, alcanza con `requests`. El rol arquitectónico (HTML→Markdown reducido) se mantiene. |
| Sustituir ScrapeGraphAI por llamada HTTP directa a Gemini | Mismo razonamiento: la librería abstrae una API que podemos llamar directamente, evitando dependencias y opacidad. |
| Recortar el HTML al `<article class="product_page">` antes de convertir | Bajamos drásticamente los tokens enviados al LLM (descartamos header/footer/menús). |
| `response_mime_type='application/json'` | Gemini soporta forzar la salida a JSON válido. Evita parseo de texto libre. |
| `temperature=0` | Determinismo: misma entrada → misma salida. Crítico para extracción de datos. |
| Pasar el **Markdown** como `source` (no la URL) al LLM | Lo pide el enunciado. Además abarata la llamada (menos tokens) y mejora la precisión. |
| `pd.DataFrame()` **al final**, no dentro del bucle | Mejor uso de memoria y garantiza un esquema único y consistente. |
| API Key vía variable de entorno | Nunca hardcodear secretos en notebooks. En Colab equivale a `userdata.get('GEMINI_API_KEY')`. |

## Etapa 0 — Instalación de dependencias

Si ejecutás el notebook por primera vez (en Colab o local), descomentá la siguiente celda:

In [ ]:
# %pip install requests beautifulsoup4 markdownify pandas python-dotenv

## Etapa 1 — Configuración: input, acumulador y credenciales

In [ ]:
import json
import os
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as html_to_md

### URLs de entrada y acumulador

Definimos las **5 URLs** que constituyen el INPUT del pipeline y una lista vacía que iremos llenando con un dict por libro extraído.

In [ ]:
URLS_LIBROS = [
    'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
    'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html',
    'https://books.toscrape.com/catalogue/soumission_998/index.html',
    'https://books.toscrape.com/catalogue/sharp-objects_997/index.html',
    'https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html',
]

registros: list[dict] = []  # acumulador

### Configuración del LLM y la API Key

- La API key se carga desde un archivo `.env` (ignorado por `.gitignore`) usando `python-dotenv`.
- Se configura una **lista de modelos en orden de preferencia**. Si el primero da rate limit (`429`) o falla, el código pasa **inmediatamente** al siguiente (sin esperas largas).
- Modelos del free tier de Google AI Studio usados como fallback:
  - `gemini-3.1-flash-lite` (cuota 500 RPD — prioritario por mayor margen)
  - `gemini-2.5-flash-lite` (rápido, cuota 20 RPD)
  - `gemini-3-flash` (cuota 20 RPD)
  - `gemini-2.5-flash` (cuota 20 RPD)
  - `gemini-flash-latest` (alias seguro)

**Setup**: crear `tp_scraping/.env` con `GEMINI_API_KEY=tu_clave`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Lista de modelos en orden de preferencia. Si el primero da rate limit (429),
# probamos UNA sola vez con un retry corto (3s) y si vuelve a fallar saltamos
# al siguiente modelo. Esto da resiliencia frente a las cuotas RPD/RPM del free tier.
MODELOS = [
    'gemini-3.1-flash-lite',   # 15 RPM / 500 RPD (mayor cuota diaria, prioritario)
    'gemini-2.5-flash-lite',   # 10 RPM / 20 RPD (rápido, fallback)
    'gemini-3-flash',          # 5 RPM / 20 RPD
    'gemini-2.5-flash',        # 5 RPM / 20 RPD
    'gemini-flash-latest',     # alias al modelo flash más reciente
]

def url_modelo(modelo: str) -> str:
    return f'https://generativelanguage.googleapis.com/v1beta/models/{modelo}:generateContent'

API_KEY = os.environ.get('GEMINI_API_KEY')
if not API_KEY:
    raise RuntimeError('Falta GEMINI_API_KEY. Crear archivo .env con la clave.')

print(f'Modelos configurados (en orden de fallback): {MODELOS}')

## Etapa 2 — Extracción + optimización + inferencia

### 2.1 Conversión HTML → Markdown reducido

Esta función cumple el rol de **Crawl4AI**: descargar la página y devolver una versión Markdown sin scripts, estilos ni elementos de navegación.

**Optimización clave**: antes de convertir, recortamos el HTML al contenedor del producto (`article.product_page`). Eso reduce el tamaño aprox. 10×, lo que se traduce directamente en menos tokens enviados al LLM (menor costo, menor latencia).

In [ ]:
def html_a_markdown(url: str) -> str:
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, 'html.parser')
    contenedor = soup.select_one('article.product_page') or soup

    md = html_to_md(str(contenedor), heading_style='ATX')
    md = re.sub(r'\n{3,}', '\n\n', md).strip()  # colapsamos saltos de línea sobrantes
    return md

### 2.2 Prompt declarativo

El prompt es **técnico y declarativo**. Define explícitamente:
- el **formato de entrada** (Markdown),
- el **esquema de salida** con tipos por campo (`string`, `float`, `integer`),
- **reglas de transformación** (ej.: `'In stock (22 available)'` → `22`).

In [ ]:
PROMPT_TEMPLATE = '''Sos un extractor de datos estructurados.

ENTRADA: contenido en formato Markdown de la ficha de un libro de la web books.toscrape.com.

TAREA: devolver EXCLUSIVAMENTE un objeto JSON con este esquema y tipos:
{{
  "titulo": string,    // título del libro
  "genero": string,    // categoría/género (ej: "Poetry", "Fiction")
  "precio": float,     // precio numérico, sin símbolo de moneda
  "stock":  integer    // cantidad de unidades disponibles, solo el número
}}

REGLAS:
- No incluyas comentarios, ni texto adicional, ni markdown. Solo JSON válido.
- Si "In stock (22 available)" → stock = 22.
- Si el precio es "£51.77" → precio = 51.77 (sin la libra).

CONTENIDO MARKDOWN:
{contenido}
'''

### 2.3 Llamada al LLM

Función equivalente al `SmartScraperGraph` de ScrapeGraphAI: enviar el Markdown junto con el prompt y obtener un dict.

Usamos `generationConfig.response_mime_type='application/json'` para que Gemini devuelva JSON válido garantizado, y `temperature=0` para reproducibilidad.

In [ ]:
def extraer_con_gemini(markdown: str) -> dict:
    '''
    Inferencia con failover RÁPIDO entre modelos.

    Estrategia: por cada modelo de la lista MODELOS, una sola petición
    con timeout corto. Si responde 429 (rate limit) o falla con timeout,
    esperamos solo 3 segundos y saltamos directamente al modelo siguiente.
    Esto evita quedarnos esperando minutos cuando un modelo está agotado.
    '''
    payload = {
        'contents': [{'parts': [{'text': PROMPT_TEMPLATE.format(contenido=markdown)}]}],
        'generationConfig': {
            'response_mime_type': 'application/json',
            'temperature': 0,
        },
    }

    ultimo_error = None
    for modelo in MODELOS:
        try:
            resp = requests.post(
                url_modelo(modelo),
                headers={'Content-Type': 'application/json', 'X-goog-api-key': API_KEY},
                json=payload,
                timeout=20,  # timeout corto para fallar rápido
            )
        except requests.exceptions.RequestException as e:
            print(f'   [{modelo}] error de red: {e.__class__.__name__} → siguiente modelo')
            ultimo_error = e
            time.sleep(1)
            continue

        if resp.status_code == 429:
            print(f'   [{modelo}] 429 rate limit → siguiente modelo')
            time.sleep(3)
            continue
        if resp.status_code in (404, 400):
            # Modelo no existe / no disponible para esta key.
            print(f'   [{modelo}] no disponible ({resp.status_code}) → siguiente modelo')
            continue
        if not resp.ok:
            print(f'   [{modelo}] error {resp.status_code} → siguiente modelo')
            ultimo_error = RuntimeError(f'{resp.status_code}: {resp.text[:200]}')
            continue

        # Éxito
        texto_json = resp.json()['candidates'][0]['content']['parts'][0]['text']
        print(f'   [modelo usado: {modelo}]')
        return json.loads(texto_json)

    raise RuntimeError(f'Todos los modelos fallaron. Último error: {ultimo_error}')

### 2.4 Bucle principal de ejecución

Por cada URL:
1. **Fetch + Markdown** con `html_a_markdown()`.
2. **Inferencia** con `extraer_con_gemini()`.
3. **Almacenamiento** con `.append()` en el acumulador.

In [ ]:
for i, url in enumerate(URLS_LIBROS, start=1):
    print(f'[{i}/{len(URLS_LIBROS)}] {url}')

    md_contenido = html_a_markdown(url)
    print(f'   Markdown: {len(md_contenido)} caracteres')

    registro = extraer_con_gemini(md_contenido)
    print(f'   Extraído: {registro}')

    registros.append(registro)
    time.sleep(1)  # pausa breve para respetar el RPM

## Etapa 3 — Transformación a DataFrame y validación

Convertimos la lista de dicts a DataFrame **una sola vez** y revisamos calidad de los datos inferidos.

In [ ]:
df = pd.DataFrame(registros)
df

### 3.1 Funciones de limpieza defensiva

Aunque el LLM en condiciones normales devuelve enteros y floats, podría ocasionalmente devolver strings como `'In stock (22 available)'` o `'£51.77'`. Definimos funciones que aplican una transformación robusta para cualquiera de los dos casos.

In [ ]:
def limpiar_stock(valor) -> int | None:
    '''Extrae el primer número entero de cualquier representación.'''
    if isinstance(valor, int):
        return valor
    if valor is None:
        return None
    match = re.search(r'\d+', str(valor))
    return int(match.group()) if match else None


def limpiar_precio(valor) -> float | None:
    '''Quita símbolos de moneda (£, $, €) y convierte a float.'''
    if isinstance(valor, (int, float)):
        return float(valor)
    if valor is None:
        return None
    limpio = re.sub(r'[^\d.]', '', str(valor))
    return float(limpio) if limpio else None

### 3.2 Aplicar limpieza y forzar tipos

Aseguramos el esquema final antes de exportar.

In [ ]:
df['precio'] = df['precio'].map(limpiar_precio)
df['stock'] = df['stock'].map(limpiar_stock)
df = df.astype({'titulo': 'string', 'genero': 'string'})

print('Nulos por columna:')
print(df.isna().sum())

print('\nTipos de datos:')
print(df.dtypes)

### 3.3 Exportación final a CSV

In [ ]:
df.to_csv('actividad_2_libros.csv', index=False, encoding='utf-8-sig')
print(f'✔ Dataset guardado: actividad_2_libros.csv ({len(df)} registros)')
df

## Conclusión

Recorrimos las **3 etapas** del pipeline ETL pedido en el enunciado:

1. **Configuración** (URLs, acumulador, credenciales seguras).
2. **Extracción + optimización** (HTML → Markdown reducido) **+ inferencia LLM** (Markdown → JSON).
3. **Transformación + validación** (lista → DataFrame → CSV con tipos garantizados).

La fase de **load (carga)** real consistiría en insertar el CSV en la base de datos / data warehouse de la empresa.